# SafeRide - train on Colab (free GPU)
**Runtime -> Change runtime type -> T4 GPU** pehle select karo.

In [ ]:
!nvidia-smi | head -n 15

## 1. Project upload karo
`saferide.zip` Colab mein upload karo (left sidebar -> Files), phir niche wala cell chalao.

In [ ]:
!unzip -q -o saferide.zip -d /content
%cd /content/saferide
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. Dataset (Kaggle)
Kaggle -> Settings -> API -> *Create New Token* -> `kaggle.json` upload karo.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json select karo
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d andrewmvd/helmet-detection -p raw --unzip

In [ ]:
!python scripts/voc_to_yolo.py --src raw --out datasets/helmet
!cat datasets/helmet/data.yaml

**Dhyan se dekho:** upar printed class names mein se jo 'no helmet' wali class hai, wo `configs/project.yaml` ke `violation_classes` mein honi chahiye.

## 3. Train

In [ ]:
!python -m saferide.train --data datasets/helmet/data.yaml --model yolov8n.pt --epochs 50 --batch 16 --name helmet_v8n

## 4. ONNX export + latency/mAP trade-off

In [ ]:
!python -m saferide.export --weights runs/helmet_v8n/weights/best.pt --sizes 320 416 640
!python -m saferide.tradeoff --weights runs/helmet_v8n/weights/best.pt --data datasets/helmet/data.yaml --sizes 320 416 640

> Note: Colab CPU numbers aur tumhare laptop ke numbers alag honge. Resume mein wahi numbers daalo jo tumne khud run kiye, aur hardware ka naam bhi likho.

In [ ]:
from IPython.display import Image
Image('results/tradeoff.png')

## 5. Results download karo

In [ ]:
!zip -qr saferide_artifacts.zip models results runs/helmet_v8n/results.png runs/helmet_v8n/confusion_matrix.png runs/helmet_v8n/weights/best.pt
from google.colab import files
files.download('saferide_artifacts.zip')